# 01 — Zero-shot eval + submission (Qwen3-14B, vLLM)

Runtime: **A100**. Chạy từ trên xuống. Kết quả (log, submission) được lưu vào `DRIVE_ROOT/runs/`.

1. Đo holdout PA trên train (6 shop của `configs/holdout.json`) → chọn thinking on/off, k, τ.
2. Chạy public_test với cấu hình tốt nhất → `submission.jsonl` → upload lên portal (tối đa 5 lần/ngày).

In [ ]:
# === Setup (chạy mỗi lần mở runtime) ===
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/DataMate_ZAC2026'   # thư mục Drive chung (Add shortcut to My Drive nếu được share)
REPO_BRANCH = 'feat/slm-agent'
import os, subprocess
if not os.path.exists('/content/AI-Challenge'):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, 'https://github.com/VTin206/AI-Challenge.git', '/content/AI-Challenge'], check=True)
%cd /content/AI-Challenge
!git pull -q
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SPLITS = ['train', 'public_test']
# === Giải nén dữ liệu từ Drive vào /content/data (đĩa local, nhanh hơn Drive) ===
import zipfile, os
os.makedirs('/content/data', exist_ok=True)
for split in SPLITS:
    if not os.path.exists(f'/content/data/{split}'):
        zipfile.ZipFile(f'{DRIVE_ROOT}/data/{split}/{split}.zip').extractall('/content/data')
!ls /content/data

In [ ]:
!pip install -q -e . vllm

## A. Holdout (train) — k=5, thinking off
Đổi `RUN` để không ghi đè. `--resume` để chạy tiếp nếu runtime bị ngắt.

In [ ]:
RUN = 'holdout_k5_nothink'
!python scripts/run_split.py --split /content/data/train --out {DRIVE_ROOT}/runs/{RUN} \
    --config configs/slm.json --holdout configs/holdout.json --tau 0 --cache-dir /content/cache --resume

In [ ]:
!python scripts/evaluate.py --predictions {DRIVE_ROOT}/runs/{RUN}/submission.jsonl --gold /content/data/train/answers.jsonl \
    --questions /content/data/train/questions.jsonl --only-predicted | head -60
!python scripts/tune_tau.py --log {DRIVE_ROOT}/runs/{RUN}/log.jsonl --gold /content/data/train/answers.jsonl

### (tuỳ chọn) thinking on — so sánh PA và thời gian/câu (`sec_per_question` trong log)

In [ ]:
RUN = 'holdout_k5_think'
!python scripts/run_split.py --split /content/data/train --out {DRIVE_ROOT}/runs/{RUN} \
    --config configs/slm.json --holdout configs/holdout.json --tau 0 --thinking --cache-dir /content/cache --resume
!python scripts/tune_tau.py --log {DRIVE_ROOT}/runs/{RUN}/log.jsonl --gold /content/data/train/answers.jsonl

In [ ]:
import json, pandas as pd
log = [json.loads(l) for l in open(f'{DRIVE_ROOT}/runs/{RUN}/log.jsonl', encoding='utf-8')]
print('s/question:', sum(r['sec_per_question'] for r in log) / len(log))
print('forced/failed samples:', sum(s['error'] is not None for r in log for s in r['samples']) / sum(len(r['samples']) for r in log))

## B. Public test → submission
Đặt `TAU` (và `THINK`) theo kết quả tune ở trên. Log được lưu cùng submission để phân tích sau.

In [ ]:
TAU = 0.6
THINK = '--no-thinking'   # hoặc '--thinking'
RUN = 'public_run1'   # ĐỔI TÊN cho mỗi cấu hình mới; giữ nguyên tên khi chạy lại sau mất kết nối (--resume)
!python scripts/run_split.py --split /content/data/public_test --out {DRIVE_ROOT}/runs/{RUN} \
    --config configs/slm.json --tau {TAU} {THINK} --cache-dir /content/cache --resume

In [ ]:
import csv, os, json
sub = f'{DRIVE_ROOT}/runs/{RUN}/submission.jsonl'
rows = [json.loads(l) for l in open(sub, encoding='utf-8')]
print(len(rows), 'answers;', sum(r['type'] == 'abstain' for r in rows), 'abstain')
log_csv = f'{DRIVE_ROOT}/submissions_log.csv'
new = not os.path.exists(log_csv)
with open(log_csv, 'a', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    if new: w.writerow(['run', 'tau', 'thinking', 'n', 'abstain', 'holdout_pa', 'portal_pa', 'note'])
    w.writerow([RUN, TAU, THINK, len(rows), sum(r['type'] == 'abstain' for r in rows), '', '', ''])
print('Upload file này lên portal:', sub)